IMPORT LIBRARIES

In [3]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

LOAD DATASET

In [4]:
df = pd.read_csv('/content/spotify_tracks.csv')

In [5]:
df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [6]:
df.shape
df.columns

Index(['track_id', 'artists', 'album_name', 'track_name', 'popularity',
       'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness',
       'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'track_genre'],
      dtype='object')

DATA CLEANING

In [7]:
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
df.isnull().sum()

,0
track_id,0
artists,0
album_name,0
track_name,0
popularity,0
duration_ms,0
explicit,0
danceability,0
energy,0
key,0


Select Features for Recommendation

In [8]:
features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'popularity'
]

Normalize Features

In [9]:
scaler = MinMaxScaler()

scaled_features = scaler.fit_transform(df[features])

Build Recommendation Model

In [10]:
from sklearn.neighbors import NearestNeighbors

model = NearestNeighbors(
    n_neighbors=6,
    metric='cosine',
    algorithm='brute'
)

model.fit(scaled_features)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=6)

Create Song Index

In [18]:
indices = pd.Series(df.index, index=df['track_name'])

Recommendation Function

In [19]:
def recommend_music(song_name, top_n=5):

    if song_name not in indices.index:
        return "Song not found"

    # Handle duplicate song names
    idx = indices[song_name]

    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    song_vector = scaled_features[idx].reshape(1, -1)

    distances, neighbors = model.kneighbors(
        song_vector,
        n_neighbors=top_n + 1
    )

    song_indices = neighbors[0][1:]

    return df[
        ['track_name', 'artists', 'track_genre']
    ].iloc[song_indices]

In [20]:
recommend_music('Hold On')

,track_name,artists,track_genre
34817,Follow The Sun,Xavier Rudd,folk
88963,Follow The Sun,Xavier Rudd,reggae
103501,If I Ain't Got You,Alicia Keys,soul
65335,The Truth Untold (feat. Steve Aoki),BTS;Steve Aoki,k-pop
102510,It's You,Sezairi,songwriter


In [21]:
recommend_music('Comedy')

,track_name,artists,track_genre
99152,Comedy,Gen Hoshino,singer-songwriter
0,Comedy,Gen Hoshino,acoustic
102151,Comedy,Gen Hoshino,songwriter
44353,Jumper - 1998 Edit,Third Eye Blind,grunge
5688,馬と鹿,Kenshi Yonezu,anime


User Input Interactive

In [22]:
song = input("Enter song name: ")

recommend_music(song)

Enter song name: best


'Song not found'

Recommendation Distance

In [24]:
def recommend_music(song_name, top_n=5):

    if song_name not in indices.index:
        return "Song not found"

    idx = indices[song_name]

    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    song_vector = scaled_features[idx].reshape(1,-1)

    distances, neighbors = model.kneighbors(
        song_vector,
        n_neighbors=top_n+1
    )

    song_indices = neighbors[0][1:]
    similarity_scores = 1 - distances[0][1:]

    results = df[
        ['track_name','artists','track_genre']
    ].iloc[song_indices].copy()

    results['Similarity'] = similarity_scores

    return results.reset_index(drop=True)

In [25]:
recommend_music('Hold On')

,track_name,artists,track_genre,Similarity
0,Follow The Sun,Xavier Rudd,folk,0.995743
1,Follow The Sun,Xavier Rudd,reggae,0.995743
2,If I Ain't Got You,Alicia Keys,soul,0.995647
3,The Truth Untold (feat. Steve Aoki),BTS;Steve Aoki,k-pop,0.995600
4,It's You,Sezairi,songwriter,0.995533
